In [1]:
!pip install psutil ipywidgets plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 466.1 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 2.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 10.7 MB/s eta 0:00:00 0:00:01m


In [ ]:
import psutil
import time
from IPython.display import display, clear_output
import ipywidgets as widgets
import plotly.graph_objects as go
from datetime import datetime

class SystemMonitor:
    def __init__(self, update_interval=2.0):
        self.update_interval = update_interval
        self.history = {'time': [], 'cpu': [], 'mem': [], 'swap': []}
        self.max_points = 60  # Хранить последние 60 точек (~2 минуты при интервале 2 сек)
        
        # Виджеты
        self.output = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '10px'})
        self.stop_button = widgets.Button(description="⏹️ Stop Monitoring", button_style='danger')
        self.stop_button.on_click(self._stop)
        self.running = False
        
    def _get_stats(self):
        return {
            'cpu_percent': psutil.cpu_percent(interval=0.1),
            'cpu_per_core': psutil.cpu_percent(interval=0.1, percpu=True),
            'mem_percent': psutil.virtual_memory().percent,
            'mem_used_gb': psutil.virtual_memory().used / 1024**3,
            'mem_total_gb': psutil.virtual_memory().total / 1024**3,
            'swap_percent': psutil.swap_memory().percent,
            'swap_used_gb': psutil.swap_memory().used / 1024**3,
            'disk_io': psutil.disk_io_counters(),
            'net_io': psutil.net_io_counters(),
            'processes': len(psutil.pids())
        }
    
    def _update_history(self, stats):
        now = datetime.now()
        self.history['time'].append(now)
        self.history['cpu'].append(stats['cpu_percent'])
        self.history['mem'].append(stats['mem_percent'])
        self.history['swap'].append(stats['swap_percent'])
        
        # Ограничение истории
        if len(self.history['time']) > self.max_points:
            for key in self.history:
                self.history[key] = self.history[key][-self.max_points:]
    
    def _render(self, stats):
        with self.output:
            clear_output(wait=True)
            
            # Основные метрики
            print(f"{'='*70}")
            print(f"📊 Системная нагрузка | Обновлено: {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*70}")
            print(f"🧠 CPU:  {stats['cpu_percent']:5.1f}%  [{'█' * int(stats['cpu_percent']/5):20s}]")
            print(f"   Ядра: " + ' | '.join([f"{c:4.0f}%" for c in stats['cpu_per_core'][:8]]))
            if len(stats['cpu_per_core']) > 8:
                print(f"         " + ' | '.join([f"{c:4.0f}%" for c in stats['cpu_per_core'][8:16]]))
            
            print(f"\n💾 RAM:   {stats['mem_percent']:5.1f}%  [{stats['mem_used_gb']:5.2f} GB / {stats['mem_total_gb']:5.2f} GB]")
            print(f"         [{'█' * int(stats['mem_percent']/5):20s}]")
            
            if stats['swap_percent'] > 0:
                print(f"🔄 Swap:  {stats['swap_percent']:5.1f}%  [{stats['swap_used_gb']:5.2f} GB]")
            
            print(f"\n⚡ Дисковый I/O:  read={stats['disk_io'].read_bytes/1024**2:.1f} MB  write={stats['disk_io'].write_bytes/1024**2:.1f} MB")
            print(f"🌐 Сеть:          sent={stats['net_io'].bytes_sent/1024**2:.1f} MB  recv={stats['net_io'].bytes_recv/1024**2:.1f} MB")
            print(f"mPid: {stats['processes']:,} активных процессов")
            print(f"{'='*70}")
            
            # График истории
            fig = go.Figure()
            fig.add_trace(go.Scatter(x=self.history['time'], y=self.history['cpu'], 
                                    mode='lines', name='CPU %', line=dict(color='red')))
            fig.add_trace(go.Scatter(x=self.history['time'], y=self.history['mem'], 
                                    mode='lines', name='RAM %', line=dict(color='blue')))
            fig.add_trace(go.Scatter(x=self.history['time'], y=self.history['swap'], 
                                    mode='lines', name='Swap %', line=dict(color='orange', dash='dot')))
            
            fig.update_layout(
                title='История нагрузки (последние 2 минуты)',
                xaxis_title='Время',
                yaxis_title='%',
                height=300,
                margin=dict(l=20, r=20, t=40, b=20),
                hovermode='x unified'
            )
            fig.show()
    
    def _stop(self, _):
        self.running = False
        print("⏹️ Мониторинг остановлен")
    
    def start(self):
        self.running = True
        print("▶️ Запуск мониторинга (нажмите кнопку 'Stop' для остановки)...")
        display(widgets.VBox([self.output, self.stop_button]))
        
        while self.running:
            stats = self._get_stats()
            self._update_history(stats)
            self._render(stats)
            time.sleep(self.max(self.update_interval, 0.5))
    
    @staticmethod
    def max(a, b):
        return a if a > b else b

# Запуск мониторинга
monitor = SystemMonitor(update_interval=2.0)
monitor.start()

▶️ Запуск мониторинга (нажмите кнопку 'Stop' для остановки)...
